# Hypoxia signature model — single-dataset discovery + scorer\n\nBuilt incrementally per `CLAUDE.md`. Scope for this session: seed list → discovery (Eqs 1, 2, 5 + Monte Carlo) → scorer (Eq 7 + HS). Adapters, Cox validation, and sigQC are out of scope here."

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## Seed genes (Buffa 2010, HNSCC training network, set A)

The ten seed genes exactly as printed in the paper, hardcoded — not a file to
load. One of the ten (`AK3L1`) was later renamed by HGNC to `AK4`; modern
datasets use the new symbol, so we keep both forms and resolve at lookup time
against whatever gene index the actual dataset has.

In [ ]:
# Ten seed genes, literal names as printed in Buffa et al. 2010.
SEED_GENES = [
    "ADM", "AK3L1", "BNIP3", "CA9", "ENO1",
    "HK2", "LDHA", "PGK1", "SLC2A1", "VEGFA",
]

# AK3L1 -> AK4 was a formal HGNC rename, not a casual alias. Exact-string
# matching against a modern dataset (Ensembl/GEO/TCGA-annotated) will miss
# AK3L1 entirely and silently drop that seed unless we also try AK4.
SEED_ALIASES = {
    "AK3L1": "AK4",
}


def resolve_seed_genes(seed_genes: list[str], available_genes) -> dict[str, str]:
    """Map each seed (paper name) to whichever symbol is present in `available_genes`.

    Tries the literal paper name first, then its known alias. Seeds matching
    neither are left out of the returned dict -- callers should check for
    missing seeds rather than assume all ten resolve.
    """
    available = set(available_genes)
    resolved = {}
    for seed in seed_genes:
        if seed in available:
            resolved[seed] = seed
        elif seed in SEED_ALIASES and SEED_ALIASES[seed] in available:
            resolved[seed] = SEED_ALIASES[seed]
    return resolved